In [0]:
!pip install opencv-python

With Parallel Processing

In [0]:
import os
import os.path as osp
import cv2
import json
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

def normalize_dbfs_path(path):
    """Ensure the path starts with '/dbfs/'."""
    if path.startswith("/dbfs/"):
        return path
    else:
        return os.path.join("/dbfs", path.lstrip("/"))

def crop_and_save(task):
    """
    Processes a single frame for one object.
    
    task: dict with keys:
       - frame_path: full path to the original image (should be normalized)
       - annotation: dict with keys "bounding_box" (list [x, y, w, h]) and "contour" (polygon points)
       - output_folder: directory where cropped image will be saved (should be normalized)
       - object_name: string
       - target_size: tuple (width, height) for resizing (e.g., (224,224))
       - fill_color: tuple (B, G, R) for background fill outside the contour.
    """
    try:
        # Load the original image.
        img = cv2.imread(task["frame_path"])
        if img is None:
            print(f"Failed to load image: {task['frame_path']}")
            return None

        # Get the bounding box and crop the region.
        bbox = task["annotation"].get("bounding_box", [0, 0, 0, 0])
        x, y, w, h = bbox
        cropped = img[y:y+h, x:x+w].copy()
        if cropped.size == 0:
            print(f"Empty crop for {task['frame_path']} with bbox {bbox}")
            return None

        # Create an empty mask for the cropped region.
        mask = np.zeros((h, w), dtype=np.uint8)
        contour = task["annotation"].get("contour", None)
        if contour is not None and len(contour) > 0:
            if isinstance(contour[0][0], (int, float)):
                cnt = np.array(contour, dtype=np.int32) - np.array([x, y])
                cnt = cnt.reshape((-1, 1, 2))
                cv2.fillPoly(mask, [cnt], 255)
            else:
                pts_list = []
                for cnt in contour:
                    cnt_arr = np.array(cnt, dtype=np.int32) - np.array([x, y])
                    pts_list.append(cnt_arr.reshape((-1, 1, 2)))
                cv2.fillPoly(mask, pts_list, 255)
        else:
            mask[:] = 255

        # Fill the background (outside the contour) with the specified fill color.
        fill_color = task.get("fill_color", (0, 0, 255))
        mask_inv = cv2.bitwise_not(mask)
        background = np.full(cropped.shape, fill_color, dtype=np.uint8)
        cropped_masked = cv2.bitwise_and(cropped, cropped, mask=mask) + \
                         cv2.bitwise_and(background, background, mask=mask_inv)

        # Resize the cropped image to the target size.
        target_size = task.get("target_size", (224, 224))
        resized = cv2.resize(cropped_masked, target_size)

        # Build the output filename: "<frame_basename>_<object_name>.jpg"
        frame_base = osp.splitext(osp.basename(task["frame_path"]))[0]
        output_filename = f"{frame_base}_{task['object_name']}.jpg"
        out_path = osp.join(task["output_folder"], output_filename)

        if cv2.imwrite(out_path, resized):
            with open(out_path, "rb") as f:
                os.fsync(f.fileno())
            print(f"Saved cropped frame: {out_path}")
        else:
            print(f"Failed to write cropped frame: {out_path}")
        return out_path
    except Exception as e:
        print(f"Error processing {task['frame_path']} for {task['object_name']}: {e}")
        return None

def crop_objects_from_annotations(annotations_json_path, frames_folder, output_base_folder,
                                  target_size=(224,224), fill_color=(0,0,255), max_workers=8):
    """
    Processes a single annotation JSON file for one object.
    
    Parameters:
      - annotations_json_path (str): Path to the annotation JSON file.
      - frames_folder (str): Path to the folder containing original frames.
      - output_base_folder (str): Base folder to save cropped images; a subfolder will be created per object.
      - target_size (tuple): Size to which the cropped images are resized.
      - fill_color (tuple): BGR color for background fill outside the object.
      - max_workers (int): Number of parallel processes.
      
    Returns:
      A dictionary mapping the object name to a list of saved file paths.
    """
    frames_folder = normalize_dbfs_path(frames_folder)
    output_base_folder = normalize_dbfs_path(output_base_folder)
    os.makedirs(output_base_folder, exist_ok=True)
    
    with open(annotations_json_path, "r") as f:
        annotations = json.load(f)
    
    base_name = osp.basename(annotations_json_path)
    object_name = base_name.replace("annotations_", "").replace(".json", "")
    
    # Create an output folder for this object directly under output_base_folder.
    output_folder = osp.join(output_base_folder, object_name)
    os.makedirs(output_folder, exist_ok=True)
    
    tasks = []
    for frame_name, ann_data in annotations.items():
        task = {
            "frame_path": osp.join(frames_folder, frame_name),
            "annotation": ann_data,
            "output_folder": output_folder,
            "object_name": object_name,
            "target_size": target_size,
            "fill_color": fill_color
        }
        tasks.append(task)
    
    saved_files = []
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(crop_and_save, task): task for task in tasks}
        for future in as_completed(futures):
            res = future.result()
            if res is not None:
                saved_files.append(res)
    return {object_name: saved_files}

def process_multiple_annotations(annotations_folder, frames_folder, output_base_folder,
                                 target_size=(224,224), fill_color=(0,0,255), max_workers=8):
    """
    Processes all annotation JSON files found in annotations_folder.
    
    Parameters:
      - annotations_folder (str): Folder containing annotation JSON files.
      - frames_folder (str): Folder containing original frames.
      - output_base_folder (str): Base folder where cropped images will be saved.
      
    Returns:
      A dictionary mapping each object name to its list of cropped file paths.
    """
    annotations_folder = normalize_dbfs_path(annotations_folder)
    ann_files = [osp.join(annotations_folder, f) for f in os.listdir(annotations_folder)
                 if f.startswith("annotations_") and f.endswith(".json")]
    print(f"Found {len(ann_files)} annotation JSON files in {annotations_folder}.")
    
    all_results = {}
    for ann_file in ann_files:
        print(f"Processing annotation file: {ann_file}")
        result = crop_objects_from_annotations(ann_file, frames_folder, output_base_folder,
                                               target_size, fill_color, max_workers)
        print(f"Finished processing annotation file: {ann_file}")
        all_results.update(result)
    return all_results

def process_all_subfolders(annotations_root, frames_root, output_root,
                           target_size=(224,224), fill_color=(0,0,255), max_workers=8):
    """
    1. For each subfolder (e.g., 'video1', 'video2', etc.) in annotations_root,
       find annotation JSON files.
    2. Match the corresponding frames subfolder in frames_root.
    3. Produce cropped images in output_root directly (one folder per object),
       without nesting under a videoxx folder.
    """
    annotations_root = normalize_dbfs_path(annotations_root)
    frames_root = normalize_dbfs_path(frames_root)
    output_root = normalize_dbfs_path(output_root)
    
    subfolders = [d for d in os.listdir(annotations_root) 
                  if osp.isdir(osp.join(annotations_root, d))]
    print(f"Found subfolders in annotations root: {subfolders}")
    
    all_results = {}
    for subfolder in subfolders:
        ann_subfolder = osp.join(annotations_root, subfolder)
        frames_subfolder = osp.join(frames_root, subfolder)
        # Instead of creating an intermediate subfolder in the output,
        # we use the top-level output_root directly.
        if not osp.exists(frames_subfolder):
            print(f"Warning: frames folder not found for subfolder '{subfolder}': {frames_subfolder}")
            continue
        
        print(f"Processing subfolder '{subfolder}' ...")
        sub_results = process_multiple_annotations(
            ann_subfolder,
            frames_subfolder,
            output_root,   # <-- Pass the top-level output root directly
            target_size=target_size,
            fill_color=fill_color,
            max_workers=max_workers
        )
        print(f"Finished processing subfolder '{subfolder}'.")
        all_results.update(sub_results)
    return all_results


# Main

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    
    parser.add_argument("--annotations_root",
                        default="/xxx/Eem/Mask coordinates/Eem_ch04_0619_060343_235956/", 
                        help="Root directory containing subfolders of annotation JSON files (e.g., video1, video2, ...)") # Path to the annotation file from SAMURAI
    parser.add_argument("--frames_root",
                        default="/xxx/Eem/Decoded Frames/Eem_ch04_0619_060343_235956/", 
                        help="Root directory containing subfolders of frames (matching subfolder names).") # Path to the decoded frames
    parser.add_argument("--output_root",
                        default="/xxx/Eem/Cropped Frames/Eem_ch04_0619_060343_235956/", 
                        help="Root directory where cropped images will be saved (one subfolder per object)") # Path to the cropped frames
    parser.add_argument("--target_size", default="224,224",
                        help="Target size as 'width,height'. Default is 224,224")
    parser.add_argument("--fill_color", default="0,0,255",
                        help="Fill color (B,G,R) as comma-separated string. Default is 0,0,255 (bright red)")
    parser.add_argument("--max_workers", default=8, type=int, 
                        help="Number of parallel workers to use") # Number of workers for parallel processing, adjust based on your cluster capability
    
    args, unknown = parser.parse_known_args()
    
    args.target_size = tuple(map(int, args.target_size.split(",")))
    args.fill_color = tuple(map(int, args.fill_color.split(",")))
    
    results = process_all_subfolders(
        annotations_root=args.annotations_root,
        frames_root=args.frames_root,
        output_root=args.output_root,
        target_size=args.target_size,
        fill_color=args.fill_color,
        max_workers=args.max_workers
    )
    
    print("All subfolders processed. Cropped files:")
    print(results)